In [ ]:
!pip -q install pyngrok

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import socket
import subprocess
import time
from urllib.request import urlopen

os.environ["OLLAMA_CONTEXT_LENGTH"] = "8192"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

if "server" in globals() and server.poll() is None:
    raise RuntimeError("Ollama is already running; reuse the existing server.")

with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", 11434)) == 0:
        raise RuntimeError("Port 11434 is occupied; inspect the existing server.")

log = open("/kaggle/working/ollama.log", "w")

server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError("Ollama exited; inspect /kaggle/working/ollama.log")
    try:
        with urlopen("http://127.0.0.1:11434/api/version", timeout=1):
            break
    except OSError:
        time.sleep(1)
else:
    raise TimeoutError("Ollama did not become ready")

In [ ]:
subprocess.run(["ollama", "pull", "qwen3.6:35b"], check=True)
subprocess.run(["ollama", "show", "qwen3.6:35b"], check=True)

In [ ]:
!pip -q install openai

In [ ]:
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

conf.get_default().config_version = "2"
conf.get_default().auth_token = secrets.get_secret("NGROK_AUTH_TOKEN")

password = secrets.get_secret("OLLAMA_TUNNEL_PASSWORD")

tunnel = ngrok.connect(
    11434,
    "http",
    auth=f"peregrine:{password}",
    host_header="localhost:11434",
    schemes=["https"],
)

print("Your Ollama API URL:")
print(tunnel.public_url + "/v1")